# Part 2 — Front / Back Analysis with HDBSCAN Clustering

This notebook reproduces and extends the experiment page's directional analysis pipeline:

1. **Deduplicate** spots to remove duplicates from overlapping WSPR windows
2. **Classify** each spot as *front* or *back* based on bearing vs bin azimuth
3. **Find paired stations** — RX stations observed in both orientations
4. **Three F/B estimators** — from simple sector medians to robust HDBSCAN cluster analysis
5. **HDBSCAN clustering** — group paired stations by propagation characteristics
6. **Hodges-Lehmann estimator** — robust typical shift with bootstrap confidence intervals
7. **Cluster recommendation** — composite score with sign-aware bonus
8. **Sensitivity analysis** across bin widths, distance, and time clustering

> Run **Part 1** first to load the data.  If you haven't, the cell below will load it independently.

## 1 · Setup & data loading

This cell loads libraries and data.  scikit-learn provides HDBSCAN clustering.

In [ ]:
import warnings, micropip
warnings.simplefilter('ignore', DeprecationWarning)
await micropip.install(['plotly', 'jinja2', 'nbformat', 'scikit-learn'])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from io import StringIO
from pathlib import Path
import math

def _find(name):
    for p in [Path(name), Path('/'+name), Path('files/'+name), Path('/files/'+name)]:
        if p.exists(): return p
    return None

csv_path = _find('experiment.csv')
if csv_path is None:
    raise RuntimeError('experiment.csv not found — relaunch from the dashboard.')

df = pd.read_csv(csv_path, dtype='string')
for c in ['snr','distance','distance_miles','distance_km','antenna_azimuth','beacon_bin_azimuth','bearing','rx_lat','rx_lon']:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
for c in ['timestamp_utc','time']:
    if c in df.columns: df[c] = pd.to_datetime(df[c], errors='coerce', utc=True)
if 'snr' in df.columns:
    df['snr'] = np.round(df['snr']).astype('Int64').astype('float64')

meta_path = _find('qth_metadata.csv')
distance_units = 'km'
if meta_path:
    meta = pd.read_csv(meta_path)
    if len(meta) and str(meta.iloc[0].get('distance_units','')).strip().lower() == 'mi':
        distance_units = 'mi'
if distance_units == 'mi' and 'distance_miles' in df.columns:
    df['distance'] = df['distance_miles']
elif 'distance_km' in df.columns:
    df['distance'] = df['distance_km']

run_ledger = None
run_role_map = {}
rl_path = _find('run_ledger.csv')
if rl_path:
    run_ledger = pd.read_csv(rl_path)
    if {'run_key','role'}.issubset(run_ledger.columns):
        tmp = run_ledger[['run_key','role']].dropna().astype(str)
        tmp['role'] = tmp['role'].str.lower()
        run_role_map = dict(zip(tmp['run_key'], tmp['role']))

print(f'{len(df)} spots loaded, {len(run_role_map)} run roles from ledger')

## 2 · Deduplicate spots

WSPR can report the same spot multiple times (network retransmissions, overlapping windows).
We keep the first occurrence using a composite key: time + receiver + band + SNR + distance.

In [ ]:
dedup_cols = [c for c in ['time','rxCall','rxGrid','band','snr','distance','bearing','run_key','pair_id'] if c in df.columns]
work = df.sort_values('time' if 'time' in df.columns else dedup_cols[0]).drop_duplicates(subset=dedup_cols, keep='first').copy()
removed = len(df) - len(work)
print(f'Before: {len(df)} → After: {len(work)} ({removed} duplicates removed, {removed/max(len(df),1)*100:.1f}%)')

## 3 · Classify front / back orientation

Each spot's bearing is compared to the **bin azimuth** (the direction you chose on the experiment page).
Spots closer to the bin azimuth are classified as *front*; spots closer to bin + 180° are *back*.

The classification uses a priority chain:
1. **Run-level rule:** Compare each run's median `antenna_azimuth` against `beacon_bin_azimuth`.
2. **Run ledger role:** The `role` column from the experiment page's run ledger.
3. **Row-level field:** `pair_role`, `role`, or `classification` from the CSV.
4. **Bearing proximity:** Angular distance to front vs back bin center.

In [ ]:
def angular_distance(a, b):
    """Shortest angular distance between two bearings (0-180)."""
    return abs((a - b + 180) % 360 - 180)

def classify_orientation(frame):
    """Assign front/back using the same priority chain as the experiment page."""
    out = frame.copy()
    out['orientation'] = pd.Series(pd.NA, index=out.index, dtype='string')

    # Method 1: run-level rule (antenna_azimuth vs bin_azimuth)
    if {'run_key','antenna_azimuth','beacon_bin_azimuth'}.issubset(out.columns):
        run_meta = (out[out['run_key'].notna()]
                    .groupby('run_key', dropna=False)
                    .agg(front_az=('antenna_azimuth','median'), bin_az=('beacon_bin_azimuth','median'))
                    .reset_index())
        if len(run_meta):
            run_meta['d_front'] = run_meta.apply(lambda r: angular_distance(r['front_az'], r['bin_az']), axis=1)
            run_meta['d_back'] = run_meta.apply(lambda r: angular_distance(r['front_az'], (r['bin_az']+180)%360), axis=1)
            run_meta['rule_orient'] = np.where(run_meta['d_front'] <= run_meta['d_back'], 'front', 'back')
            rule_map = dict(zip(run_meta['run_key'].astype(str), run_meta['rule_orient']))
            if rule_map:
                out['orientation'] = out['run_key'].astype(str).map(rule_map).astype('string')

    # Method 2: run ledger
    mask_missing = out['orientation'].isna()
    if mask_missing.any() and run_role_map and 'run_key' in out.columns:
        ledger_orient = out.loc[mask_missing, 'run_key'].astype(str).map(run_role_map)
        out.loc[mask_missing, 'orientation'] = ledger_orient.astype('string')

    # Method 3: row-level role field
    mask_missing = out['orientation'].isna()
    if mask_missing.any():
        role_col = next((c for c in ['pair_role','role','classification'] if c in out.columns), None)
        if role_col:
            txt = out.loc[mask_missing, role_col].astype(str).str.lower()
            out.loc[mask_missing, 'orientation'] = np.where(txt.str.contains('front'), 'front', 'back')

    # Method 4: bearing proximity fallback
    mask_missing = out['orientation'].isna()
    if mask_missing.any() and {'bearing','beacon_bin_azimuth'}.issubset(out.columns):
        bin_az = out.loc[mask_missing, 'beacon_bin_azimuth']
        bearing = out.loc[mask_missing, 'bearing']
        d_front = bearing.combine(bin_az, lambda b, a: angular_distance(b, a))
        d_back = bearing.combine(bin_az, lambda b, a: angular_distance(b, (a+180)%360))
        out.loc[mask_missing, 'orientation'] = np.where(d_front <= d_back, 'front', 'back')

    out['orientation'] = out['orientation'].fillna('front').astype('string')
    return out

work = classify_orientation(work)
orient_counts = work['orientation'].value_counts()
print('Orientation counts:')
print(orient_counts.to_string())

## 4 · Front vs Back SNR comparison

A side-by-side box plot comparing SNR distributions for front and back spots.
If your antenna has directional gain, the front distribution should shift higher.

In [ ]:
if 'snr' in work.columns and work['snr'].notna().any():
    fb = work[work['orientation'].isin(['front','back'])].copy()
    colors = {'front': '#3b82f6', 'back': '#f97316'}
    fig = px.box(fb, x='orientation', y='snr', color='orientation',
                 color_discrete_map=colors,
                 title='SNR by Orientation',
                 labels={'snr':'SNR (dB)','orientation':'Orientation'},
                 points='outliers')
    fig.update_layout(height=400, showlegend=False)
    fig.show()

    front_med = fb.loc[fb['orientation']=='front','snr'].median()
    back_med = fb.loc[fb['orientation']=='back','snr'].median()
    print(f'Front median: {front_med:.1f} dB  |  Back median: {back_med:.1f} dB  |  Δ = {front_med - back_med:+.1f} dB')
else:
    print('No SNR data.')

## 5 · Station-level paired comparison

The most reliable F/B evidence comes from **paired stations** — RX stations that heard you in
both front and back orientations.  For each paired station we compute:

$$\Delta_s = \text{median}(\text{SNR}_{\text{front},s}) - \text{median}(\text{SNR}_{\text{back},s})$$

The overall station-paired F/B estimate is the **median of all station deltas**, which reduces
the impact of outlier stations.  This is the **second-strongest estimator** (after HDBSCAN).

In [ ]:
def compute_paired_fb(frame, station_col='rxCall'):
    """Compute station-level F/B deltas for stations with both front and back observations."""
    sub = frame[frame['orientation'].isin(['front','back']) & frame['snr'].notna() & frame[station_col].notna()].copy()
    if sub.empty:
        return pd.DataFrame(columns=[station_col,'front_snr','back_snr','fb_delta','spot_count'])

    grouped = sub.groupby([station_col,'orientation'])['snr'].agg(['median','count']).unstack('orientation')
    if ('median','front') not in grouped.columns or ('median','back') not in grouped.columns:
        return pd.DataFrame(columns=[station_col,'front_snr','back_snr','fb_delta','spot_count'])

    paired = grouped.dropna(subset=[('median','front'), ('median','back')]).copy()
    if paired.empty:
        return pd.DataFrame(columns=[station_col,'front_snr','back_snr','fb_delta','spot_count'])

    result = pd.DataFrame({
        station_col: paired.index,
        'front_snr': paired[('median','front')].values,
        'back_snr': paired[('median','back')].values,
        'front_spots': paired[('count','front')].fillna(0).astype(int).values,
        'back_spots': paired[('count','back')].fillna(0).astype(int).values,
    })
    result['fb_delta'] = result['front_snr'] - result['back_snr']
    result['spot_count'] = result['front_spots'] + result['back_spots']
    return result.sort_values('fb_delta', ascending=False).reset_index(drop=True)

pairs = compute_paired_fb(work)
print(f'Paired stations: {len(pairs)}')
if len(pairs):
    fb_median = pairs['fb_delta'].median()
    fb_mean = pairs['fb_delta'].mean()
    fb_std = pairs['fb_delta'].std()
    print(f'Station-paired F/B estimate:  median = {fb_median:+.1f} dB  |  mean = {fb_mean:+.1f} dB  |  std = {fb_std:.1f} dB')
    pairs.head(20)
else:
    print('No paired stations found.')

### Paired station F/B waterfall chart

Each bar is one RX station.  Bars above zero heard you louder from the front;
bars below zero heard you louder from the back.  The dashed line shows the median F/B.

In [ ]:
if len(pairs):
    sorted_pairs = pairs.sort_values('fb_delta', ascending=True)
    colors = ['#3b82f6' if d >= 0 else '#f97316' for d in sorted_pairs['fb_delta']]
    fig = go.Figure(go.Bar(
        x=sorted_pairs['fb_delta'],
        y=sorted_pairs['rxCall'],
        orientation='h',
        marker_color=colors,
        text=[f'{d:+.1f} dB' for d in sorted_pairs['fb_delta']],
        textposition='outside',
        hovertemplate='%{y}<br>F/B: %{x:+.1f} dB<br>Spots: %{customdata}<extra></extra>',
        customdata=sorted_pairs['spot_count'],
    ))
    fb_med = pairs['fb_delta'].median()
    fig.add_vline(x=fb_med, line_dash='dash', line_color='#dc2626',
                  annotation_text=f'median {fb_med:+.1f} dB')
    fig.add_vline(x=0, line_color='#94a3b8', line_width=1)
    fig.update_layout(
        title='Station-Level F/B Delta (paired stations)',
        xaxis_title='F/B Delta (dB)',
        yaxis_title='RX Station',
        height=max(350, len(sorted_pairs)*28),
        margin=dict(l=120, t=50, b=40),
    )
    fig.show()
else:
    print('No paired stations to plot.')

## 6 · HDBSCAN Cluster Analysis

### Why HDBSCAN?

The station-paired F/B estimate (Section 5) treats all paired stations equally.
But in practice, stations at different distances and bearings may experience
different propagation modes (ground wave, sky wave, multi-hop).  **HDBSCAN**
(Hierarchical Density-Based Spatial Clustering of Applications with Noise)
automatically discovers groups of stations that share similar propagation characteristics.

### How it works

Each paired station becomes a point in a 4-dimensional feature space:

| Feature | What it captures |
|---------|-----------------|
| **SNR delta** | Front − back signal difference (the primary signal) |
| **sin(bearing)** | Circular bearing component (avoids 0°/360° wrapping) |
| **cos(bearing)** | Complementary circular component |
| **distance** | Path length to RX station |

HDBSCAN finds clusters of stations that are close together in this space —
meaning they have similar F/B deltas at similar bearings and distances.
Noise points (stations that don't belong to any cluster) are excluded.

### What you get

For each cluster:
- **Hodges-Lehmann estimator** — a robust "typical shift" (median of all pairwise means)
- **Bootstrap 95% CI** — uncertainty range from 2000 resamples
- **Composite score** — quality metric combining sample size, membership confidence, signal-to-scatter ratio, and **sign consistency**
- **★ Recommended cluster** — the cluster with the highest composite score

In [ ]:
from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import StandardScaler
from itertools import combinations

def hodges_lehmann(values):
    """Hodges-Lehmann estimator: median of all pairwise means.
    
    More robust than the arithmetic mean — a single outlier can't dominate.
    For n values, computes n*(n+1)/2 pairwise averages and returns their median.
    """
    vals = np.asarray(values, dtype=float)
    n = len(vals)
    if n == 0:
        return np.nan
    if n == 1:
        return float(vals[0])
    # Include self-pairs (i,i) for completeness
    pw = []
    for i in range(n):
        for j in range(i, n):
            pw.append((vals[i] + vals[j]) / 2.0)
    return float(np.median(pw))

def weighted_median(values, weights):
    """Weighted median: the value where cumulative weight reaches 50%."""
    vals = np.asarray(values, dtype=float)
    wts = np.asarray(weights, dtype=float)
    mask = np.isfinite(vals) & np.isfinite(wts) & (wts > 0)
    vals, wts = vals[mask], wts[mask]
    if len(vals) == 0:
        return np.nan
    order = np.argsort(vals)
    vals, wts = vals[order], wts[order]
    cumw = np.cumsum(wts)
    cutoff = cumw[-1] / 2.0
    idx = np.searchsorted(cumw, cutoff)
    return float(vals[min(idx, len(vals) - 1)])

def compute_hdbscan_clusters(pairs_df, work_df, min_cluster_size=3):
    """Run HDBSCAN on paired station data and score each cluster.
    
    Returns:
        points: list of dicts with per-station features
        cluster_scores: list of dicts with per-cluster metrics
        labels: cluster labels per point (-1 = noise)
    """
    if len(pairs_df) < 4:
        return [], [], np.array([])
    
    # Build feature matrix from paired stations
    station_col = 'rxCall'
    bin_az = work_df['beacon_bin_azimuth'].median() if 'beacon_bin_azimuth' in work_df.columns else 0
    
    points = []
    for _, row in pairs_df.iterrows():
        call = row[station_col]
        # Get station's typical bearing and distance
        stn_rows = work_df[work_df[station_col] == call]
        bearing = stn_rows['bearing'].median() if 'bearing' in stn_rows.columns else 0
        distance = stn_rows['distance'].median() if 'distance' in stn_rows.columns else 0
        
        bearing_rad = np.radians(bearing)
        points.append({
            'station': call,
            'snr_delta': row['fb_delta'],
            'bearing': bearing,
            'sin_bearing': np.sin(bearing_rad),
            'cos_bearing': np.cos(bearing_rad),
            'distance_km': distance if not np.isnan(distance) else 0,
            'front_count': row['front_spots'],
            'back_count': row['back_spots'],
            'n_paired': row['front_spots'] + row['back_spots'],
        })
    
    # Feature matrix: [snr_delta, sin_bearing, cos_bearing, distance_km]
    X = np.array([[p['snr_delta'], p['sin_bearing'], p['cos_bearing'], p['distance_km']] for p in points])
    
    # StandardScaler normalization
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # HDBSCAN clustering
    effective_min = max(2, min(min_cluster_size, len(points) // 2))
    clusterer = HDBSCAN(min_cluster_size=effective_min)
    labels = clusterer.fit_predict(X_scaled)
    probabilities = clusterer.probabilities_
    
    # Assign cluster labels and memberships to points
    for i, p in enumerate(points):
        p['cluster'] = int(labels[i])
        p['membership'] = float(probabilities[i])
    
    # Score each cluster
    unique_clusters = sorted(set(labels[labels >= 0]))
    cluster_scores = []
    
    for cid in unique_clusters:
        members = [p for p in points if p['cluster'] == cid]
        n = len(members)
        if n < 2:
            continue
        
        deltas = np.array([p['snr_delta'] for p in members])
        memberships = np.array([p['membership'] for p in members])
        n_paired = np.array([p['n_paired'] for p in members])
        
        # Core metrics
        median_delta = float(np.median(deltas))
        mad = float(np.median(np.abs(deltas - median_delta)))
        mean_membership = float(np.mean(memberships))
        
        # Hodges-Lehmann estimator
        hl = hodges_lehmann(deltas)
        
        # Weighted median: w_i = sqrt(n_paired_i) * membership_i
        weights = np.sqrt(n_paired.astype(float)) * memberships
        wmed = weighted_median(deltas, weights)
        
        # Bootstrap 95% CI (2000 resamples)
        rng = np.random.default_rng(seed=42)
        boot = np.array([hodges_lehmann(rng.choice(deltas, size=n, replace=True)) for _ in range(2000)])
        ci_lo = float(np.percentile(boot, 2.5))
        ci_hi = float(np.percentile(boot, 97.5))
        
        # Sign consistency
        sign_positive = float(np.mean(deltas > 0))
        
        # Composite score with sign bonus
        eps = 0.1
        signal_to_scatter = abs(median_delta) / (mad + eps)
        if sign_positive >= 0.5:
            sign_bonus = sign_positive ** 2
        else:
            sign_bonus = (1.0 - sign_positive) ** 2 * 0.25
        score = math.log(max(n, 2)) * mean_membership * signal_to_scatter * sign_bonus
        
        cluster_scores.append({
            'cluster_id': cid,
            'n_stations': n,
            'total_spots': int(sum(p['front_count'] + p['back_count'] for p in members)),
            'mean_membership': round(mean_membership, 3),
            'median_delta': round(median_delta, 2),
            'mad': round(mad, 2),
            'hodges_lehmann': round(hl, 2),
            'weighted_median': round(wmed, 2),
            'bootstrap_ci_lo': round(ci_lo, 2),
            'bootstrap_ci_hi': round(ci_hi, 2),
            'sign_positive': round(sign_positive, 3),
            'score': round(score, 3),
        })
    
    cluster_scores.sort(key=lambda c: c['score'], reverse=True)
    return points, cluster_scores, labels

# Run HDBSCAN
points, cluster_scores, cluster_labels = compute_hdbscan_clusters(pairs, work)

if cluster_scores:
    print(f'HDBSCAN found {len(set(cluster_labels[cluster_labels >= 0]))} clusters, '
          f'{(cluster_labels == -1).sum()} noise points out of {len(pairs)} stations')
    print()
    for cs in cluster_scores:
        star = ' ★' if cs == cluster_scores[0] else ''
        sign = '+' if cs['hodges_lehmann'] >= 0 else ''
        print(f"  Cluster {cs['cluster_id']}{star}: HL = {sign}{cs['hodges_lehmann']:.1f} dB "
              f"[{cs['bootstrap_ci_lo']:.1f}, {cs['bootstrap_ci_hi']:.1f}], "
              f"score = {cs['score']:.1f}, "
              f"{cs['n_stations']} stations, "
              f"sign+ = {cs['sign_positive']*100:.0f}%")
else:
    print('Not enough paired stations for HDBSCAN clustering (need ≥ 4).')

### Understanding the composite score

The composite score ranks cluster quality for F/B estimation:

$$\text{score} = \log(N) \times \bar{m} \times \frac{|\tilde{\Delta}|}{\text{MAD} + \epsilon} \times \text{sign\_bonus}$$

| Component | What it rewards |
|-----------|----------------|
| $\log(N)$ | More stations (diminishing returns) |
| $\bar{m}$ | Higher HDBSCAN membership confidence |
| $\frac{\|\tilde{\Delta}\|}{\text{MAD} + \epsilon}$ | Strong signal relative to scatter (signal-to-noise) |
| sign_bonus | Directional consistency |

**Sign bonus** is critical for directional antennas:
- If most stations agree front > back (sign+ ≥ 50%): bonus = (sign+)² — **rewards consensus**
- If most stations show back > front (sign+ < 50%): bonus = (1−sign+)² × 0.25 — **4× penalty**

This ensures a cluster where front consistently beats back (the expected behavior for a directional antenna)
is **always preferred** over an equal-magnitude cluster with the opposite sign.  Negative-delta clusters
are penalized but not zeroed — they may still be informative for sidelobe analysis.

### Understanding the Hodges-Lehmann estimator

The HL estimator computes the median of all $\binom{n}{2} + n$ pairwise averages:

$$\hat{\theta}_{HL} = \text{median}\left\{\frac{\Delta_i + \Delta_j}{2} : 1 \leq i \leq j \leq n\right\}$$

Why use it instead of the simple median?
- **Robust**: a single extreme station can't dominate (like the median)
- **More efficient**: uses information from all pairs, not just the middle value
- **Breakdown point**: 29% (can tolerate ~29% outlier stations before the estimate becomes meaningless)

The **bootstrap 95% CI** resamples the station deltas 2000 times and takes the 2.5th–97.5th percentile
of the HL estimates — giving a direct uncertainty range without distributional assumptions.

### HDBSCAN cluster scatter plot

Each dot is a paired station.  The horizontal axis is distance, the vertical axis is SNR delta
(front − back).  Colors show cluster membership; grey dots are noise (no cluster).
The ★ cluster has the highest composite score.

In [ ]:
if points and cluster_scores:
    pts_df = pd.DataFrame(points)
    pts_df['cluster_label'] = pts_df['cluster'].apply(
        lambda c: f'Cluster {c}' if c >= 0 else 'Noise')
    
    # Build color map: recommended cluster gets a special color
    recommended_id = cluster_scores[0]['cluster_id']
    unique_cids = sorted(pts_df.loc[pts_df['cluster'] >= 0, 'cluster'].unique())
    
    palette = px.colors.qualitative.Set2
    color_map = {'Noise': '#d1d5db'}
    for i, cid in enumerate(unique_cids):
        label = f'Cluster {cid}'
        color_map[label] = palette[i % len(palette)]
    
    fig = px.scatter(pts_df, x='distance_km', y='snr_delta',
                     color='cluster_label', color_discrete_map=color_map,
                     hover_name='station',
                     hover_data={'membership': ':.2f', 'bearing': ':.0f',
                                 'n_paired': True, 'cluster_label': False,
                                 'distance_km': ':.0f', 'snr_delta': ':+.1f'},
                     title='HDBSCAN Clusters: Distance vs SNR Delta',
                     labels={'distance_km': f'Distance ({distance_units})',
                             'snr_delta': 'SNR Delta (dB)',
                             'cluster_label': 'Cluster'})
    
    # Add horizontal line at y=0
    fig.add_hline(y=0, line_color='#94a3b8', line_width=1, line_dash='dot')
    
    # Add recommended cluster HL line
    best = cluster_scores[0]
    fig.add_hline(y=best['hodges_lehmann'], line_color='#dc2626', line_dash='dash',
                  annotation_text=f"★ HL = {best['hodges_lehmann']:+.1f} dB",
                  annotation_position='top right')
    
    # CI band for recommended cluster
    fig.add_hrect(y0=best['bootstrap_ci_lo'], y1=best['bootstrap_ci_hi'],
                  fillcolor='#dc2626', opacity=0.08, line_width=0,
                  annotation_text=f"95% CI [{best['bootstrap_ci_lo']:.1f}, {best['bootstrap_ci_hi']:.1f}]",
                  annotation_position='bottom right')
    
    fig.update_layout(height=500, margin=dict(t=60, b=40))
    fig.update_traces(marker=dict(size=10, line=dict(width=1, color='white')))
    fig.show()
else:
    print('No HDBSCAN clusters to plot.')

### Cluster summary table

All clusters ranked by composite score.  The ★ recommended cluster is the best candidate
for the F/B estimate.  Key columns:

- **HL (dB)**: Hodges-Lehmann F/B estimate — positive means front > back
- **95% CI**: Bootstrap confidence interval — narrower = more certain
- **Sign+**: Fraction of stations where front > back — higher = more consistent
- **Score**: Composite quality metric — higher = better

In [ ]:
if cluster_scores:
    summary_rows = []
    for cs in cluster_scores:
        star = '★ ' if cs == cluster_scores[0] else '  '
        summary_rows.append({
            '': star,
            'Cluster': cs['cluster_id'],
            'Stations': cs['n_stations'],
            'Spots': cs['total_spots'],
            'HL (dB)': f"{cs['hodges_lehmann']:+.1f}",
            '95% CI': f"[{cs['bootstrap_ci_lo']:.1f}, {cs['bootstrap_ci_hi']:.1f}]",
            'CI width': f"{cs['bootstrap_ci_hi'] - cs['bootstrap_ci_lo']:.1f}",
            'Weighted med': f"{cs['weighted_median']:+.1f}",
            'Sign+': f"{cs['sign_positive']*100:.0f}%",
            'MAD': f"{cs['mad']:.1f}",
            'Membership': f"{cs['mean_membership']:.2f}",
            'Score': f"{cs['score']:.1f}",
        })
    display(pd.DataFrame(summary_rows).set_index(''))
    
    # Print recommendation
    best = cluster_scores[0]
    print(f"\n★ Recommended: Cluster {best['cluster_id']}")
    print(f"  Hodges-Lehmann F/B = {best['hodges_lehmann']:+.1f} dB")
    print(f"  95% CI: [{best['bootstrap_ci_lo']:.1f}, {best['bootstrap_ci_hi']:.1f}] dB")
    print(f"  {best['n_stations']} stations, {best['total_spots']} total spots")
    print(f"  Sign+ = {best['sign_positive']*100:.0f}% (fraction of stations with front > back)")
    if best['sign_positive'] < 0.6:
        print(f"  ⚠ Sign+ < 60% — ambiguous direction, interpret with caution")
    ci_width = best['bootstrap_ci_hi'] - best['bootstrap_ci_lo']
    if ci_width > 8:
        print(f"  ⚠ CI width {ci_width:.1f} dB > 8 dB — high uncertainty, collect more data")
else:
    print('No cluster scores available.')

## 7 · Three F/B estimators compared

The system provides three estimators of increasing sophistication:

| Estimator | Method | Strength |
|-----------|--------|----------|
| **Simple F/B** | median(SNR_front) − median(SNR_back) | Fast but ignores station pairing |
| **Station-paired F/B** | median of per-station deltas | Accounts for pairing but not propagation modes |
| **HDBSCAN cluster F/B** | Hodges-Lehmann of best cluster | Robust, mode-aware, with uncertainty quantification |

The HDBSCAN estimate is the **primary inference metric** when sufficient paired stations exist.

In [ ]:
# Compute all three estimators
estimator_comparison = []

# 1. Simple F/B (sector medians)
front_snr = work.loc[work['orientation']=='front', 'snr'].dropna()
back_snr = work.loc[work['orientation']=='back', 'snr'].dropna()
if len(front_snr) and len(back_snr):
    simple_fb = float(front_snr.median() - back_snr.median())
    estimator_comparison.append({
        'Estimator': 'Simple F/B (sector medians)',
        'F/B (dB)': f'{simple_fb:+.1f}',
        'Uncertainty': '—',
        'Stations': f'{len(front_snr)} front / {len(back_snr)} back spots',
        'Notes': 'Weakest — no pairing, no outlier resistance',
    })

# 2. Station-paired F/B
if len(pairs):
    paired_fb = float(pairs['fb_delta'].median())
    paired_iqr = float(pairs['fb_delta'].quantile(0.75) - pairs['fb_delta'].quantile(0.25))
    estimator_comparison.append({
        'Estimator': 'Station-paired F/B (median Δ)',
        'F/B (dB)': f'{paired_fb:+.1f}',
        'Uncertainty': f'IQR = {paired_iqr:.1f} dB',
        'Stations': f'{len(pairs)} paired stations',
        'Notes': 'Pre-HDBSCAN robust estimate',
    })

# 3. HDBSCAN cluster F/B
if cluster_scores:
    best = cluster_scores[0]
    estimator_comparison.append({
        'Estimator': '★ HDBSCAN cluster F/B (Hodges-Lehmann)',
        'F/B (dB)': f"{best['hodges_lehmann']:+.1f}",
        'Uncertainty': f"95% CI [{best['bootstrap_ci_lo']:.1f}, {best['bootstrap_ci_hi']:.1f}]",
        'Stations': f"{best['n_stations']} in cluster {best['cluster_id']}",
        'Notes': f"Score {best['score']:.1f}, sign+ {best['sign_positive']*100:.0f}%",
    })

if estimator_comparison:
    display(pd.DataFrame(estimator_comparison).set_index('Estimator'))
else:
    print('No estimators could be computed.')

## 8 · HDBSCAN stability & legacy sensitivity analysis

Now that HDBSCAN is the primary estimator, the key question is: **how stable is the
HDBSCAN F/B estimate when we vary the input data?**

We test stability by re-running the full HDBSCAN pipeline on data subsets:

- **Time halves:** Early vs late — catches time-of-day drift in propagation.
- **Distance halves:** Near vs far — catches path-length–dependent mode shifts.

If the HDBSCAN HL estimate agrees across subsets (within ≈2–3 dB), the result is robust.
Large disagreements signal confounders that HDBSCAN couldn't separate.

For comparison, we also show the **legacy station-paired median** across bin-width
variations (±30° vs ±45°).  This pre-HDBSCAN estimator is weaker but provides a
cross-check: if the paired median wildly disagrees with HDBSCAN, investigate why.

In [ ]:
# --- HDBSCAN stability: re-run on data subsets ---
def hdbscan_on_subset(frame, label):
    """Re-run the full paired→HDBSCAN pipeline on a data subset."""
    sub_pairs = compute_paired_fb(frame)
    if len(sub_pairs) < 4:
        return {'Subset': label, 'Method': 'HDBSCAN', 'Stations': len(sub_pairs),
                'F/B (dB)': '—', 'Uncertainty': 'too few for clustering', 'Notes': ''}
    _, scores, _ = compute_hdbscan_clusters(sub_pairs, frame)
    if scores:
        best = scores[0]
        return {'Subset': label, 'Method': '★ HDBSCAN (HL)',
                'Stations': best['n_stations'],
                'F/B (dB)': f"{best['hodges_lehmann']:+.1f}",
                'Uncertainty': f"[{best['bootstrap_ci_lo']:.1f}, {best['bootstrap_ci_hi']:.1f}]",
                'Notes': f"score {best['score']:.1f}, sign+ {best['sign_positive']*100:.0f}%"}
    return {'Subset': label, 'Method': 'HDBSCAN', 'Stations': len(sub_pairs),
            'F/B (dB)': '—', 'Uncertainty': 'no clusters formed', 'Notes': ''}

stability_rows = []

# Full dataset HDBSCAN (reference)
if cluster_scores:
    best = cluster_scores[0]
    stability_rows.append({
        'Subset': 'All data (reference)', 'Method': '★ HDBSCAN (HL)',
        'Stations': best['n_stations'],
        'F/B (dB)': f"{best['hodges_lehmann']:+.1f}",
        'Uncertainty': f"[{best['bootstrap_ci_lo']:.1f}, {best['bootstrap_ci_hi']:.1f}]",
        'Notes': f"score {best['score']:.1f}"})

# Time subsets
time_col = 'time' if 'time' in work.columns else 'timestamp_utc'
if time_col in work.columns and work[time_col].notna().any():
    median_t = work[time_col].dropna().quantile(0.5)
    early = work[work[time_col] <= median_t]
    late = work[work[time_col] > median_t]
    stability_rows.append(hdbscan_on_subset(early, 'Early half (time)'))
    stability_rows.append(hdbscan_on_subset(late, 'Late half (time)'))

# Distance subsets
if 'distance' in work.columns and work['distance'].notna().any():
    median_d = work['distance'].quantile(0.5)
    near = work[work['distance'] <= median_d]
    far = work[work['distance'] > median_d]
    stability_rows.append(hdbscan_on_subset(near, 'Near half (distance)'))
    stability_rows.append(hdbscan_on_subset(far, 'Far half (distance)'))

if stability_rows:
    print('=== HDBSCAN Stability Across Data Subsets ===')
    display(pd.DataFrame(stability_rows).set_index('Subset'))

    # Check agreement
    hl_vals = [float(r['F/B (dB)'].replace('+','')) for r in stability_rows if r['F/B (dB)'] != '—']
    if len(hl_vals) >= 2:
        spread = max(hl_vals) - min(hl_vals)
        if spread <= 3:
            print(f'\n✓ HDBSCAN estimates agree within {spread:.1f} dB — stable result.')
        else:
            print(f'\n⚠ HDBSCAN estimates spread {spread:.1f} dB — investigate time/distance confounders.')

# --- Legacy sensitivity: station-paired median across bin widths ---
def filtered_fb(frame, half_width=45, time_cluster='all', dist_cluster='all'):
    """Apply filters then compute paired F/B (legacy estimator)."""
    sub = frame.copy()
    if half_width != 45 and {'bearing','beacon_bin_azimuth'}.issubset(sub.columns):
        bin_az = sub['beacon_bin_azimuth'].median()
        d_front = sub['bearing'].apply(lambda b: angular_distance(b, bin_az))
        d_back = sub['bearing'].apply(lambda b: angular_distance(b, (bin_az+180)%360))
        in_front = d_front <= half_width
        in_back = d_back <= half_width
        sub = sub[in_front | in_back].copy()
        sub['orientation'] = np.where(in_front[sub.index], 'front', 'back')

    time_col = 'time' if 'time' in sub.columns else 'timestamp_utc'
    if time_cluster != 'all' and time_col in sub.columns and sub[time_col].notna().any():
        median_t = sub[time_col].dropna().quantile(0.5)
        if time_cluster == 'early': sub = sub[sub[time_col] <= median_t]
        elif time_cluster == 'late': sub = sub[sub[time_col] > median_t]

    if dist_cluster != 'all' and 'distance' in sub.columns and sub['distance'].notna().any():
        median_d = sub['distance'].quantile(0.5)
        if dist_cluster == 'near': sub = sub[sub['distance'] <= median_d]
        elif dist_cluster == 'far': sub = sub[sub['distance'] > median_d]

    result = compute_paired_fb(sub)
    return {
        'pairs': len(result),
        'fb_median': float(result['fb_delta'].median()) if len(result) else np.nan,
        'fb_mean': float(result['fb_delta'].mean()) if len(result) else np.nan,
        'fb_std': float(result['fb_delta'].std()) if len(result) > 1 else np.nan,
    }

print('\n=== Legacy Sensitivity: Station-Paired Median ===')
estimators = [
    ('±45° all data', 45, 'all', 'all'),
    ('±30° all data', 30, 'all', 'all'),
    ('±45° early half', 45, 'early', 'all'),
    ('±45° late half', 45, 'late', 'all'),
    ('±45° near half', 45, 'all', 'near'),
    ('±45° far half', 45, 'all', 'far'),
]

rows = []
for label, hw, tc, dc in estimators:
    result = filtered_fb(work, half_width=hw, time_cluster=tc, dist_cluster=dc)
    rows.append({'Estimator': label, 'Paired stations': result['pairs'],
                 'F/B median (dB)': f"{result['fb_median']:+.1f}" if not np.isnan(result['fb_median']) else '—',
                 'F/B mean (dB)': f"{result['fb_mean']:+.1f}" if not np.isnan(result['fb_mean']) else '—',
                 'Std dev': f"{result['fb_std']:.1f}" if not np.isnan(result['fb_std']) else '—'})

est_df = pd.DataFrame(rows)
est_df.style.set_properties(**{'text-align':'center'}, subset=est_df.columns[1:])

### Stability comparison chart

This chart shows both the HDBSCAN HL estimates (primary) and legacy paired-median
estimates across data subsets.  The HDBSCAN bars should cluster tightly around the
reference value.  If the legacy bars diverge, it confirms why HDBSCAN is the
stronger estimator.

In [ ]:
# Combine HDBSCAN stability + legacy sensitivity into one chart
chart_data = []

# HDBSCAN values
for r in stability_rows:
    if r['F/B (dB)'] != '—':
        chart_data.append({'Label': r['Subset'], 'F/B (dB)': float(r['F/B (dB)'].replace('+','')),
                           'Method': 'HDBSCAN (HL)'})

# Legacy values
for r in rows:
    if r['F/B median (dB)'] != '—':
        chart_data.append({'Label': r['Estimator'], 'F/B (dB)': float(r['F/B median (dB)'].replace('+','')),
                           'Method': 'Paired median (legacy)'})

if chart_data:
    cdf = pd.DataFrame(chart_data)
    fig = px.bar(cdf, x='Label', y='F/B (dB)', color='Method',
                 barmode='group',
                 color_discrete_map={'HDBSCAN (HL)': '#3b82f6', 'Paired median (legacy)': '#f97316'},
                 title='F/B Estimate Stability: HDBSCAN vs Legacy',
                 text=[f'{v:+.1f}' for v in cdf['F/B (dB)']])
    fig.update_traces(textposition='outside')
    fig.add_hline(y=0, line_color='#94a3b8', line_width=1)
    if cluster_scores:
        ref = cluster_scores[0]['hodges_lehmann']
        fig.add_hline(y=ref, line_dash='dash', line_color='#dc2626',
                      annotation_text=f'Reference HL = {ref:+.1f} dB')
    fig.update_layout(height=450, margin=dict(t=60, b=80), xaxis_tickangle=-30)
    fig.show()
else:
    print('No estimator produced a result.')

## 9 · Per-run F/B

Breaking down F/B by individual run shows how the estimate varies across data collection
sessions.  Consistent F/B across runs strengthens the conclusion.

In [ ]:
run_stats = None
if 'run_key' in work.columns and work['run_key'].nunique() > 1 and 'snr' in work.columns:
    run_stats = (work[work['orientation'].isin(['front','back']) & work['snr'].notna()]
                 .groupby(['run_key','orientation'])['snr']
                 .agg(['median','count','std'])
                 .reset_index())

    if run_role_map:
        run_stats['role'] = run_stats['run_key'].astype(str).map(run_role_map).fillna('')
    else:
        run_stats['role'] = run_stats['orientation']

    run_stats.columns = ['run_key','orientation','snr_median','spot_count','snr_std','role']

    fig = px.bar(run_stats, x='run_key', y='snr_median', color='orientation',
                 color_discrete_map={'front':'#3b82f6','back':'#f97316'},
                 barmode='group',
                 title='Median SNR per Run',
                 labels={'snr_median':'Median SNR (dB)','run_key':'Run'},
                 hover_data=['spot_count','role'])
    fig.update_layout(height=400, xaxis_tickangle=-45)
    fig.show()
else:
    print('Need multiple runs with SNR data for per-run analysis.')

## 10 · Confidence assessment with quality gates

The confidence score combines multiple evidence dimensions.  When HDBSCAN cluster data
is available, the formula uses 5 components:

$$\text{confidence} = 0.30 \times \text{evidence} + 0.15 \times \text{classified} + 0.15 \times \text{overlap} + 0.25 \times \text{cluster\_quality} + 0.15 \times \text{ci\_tightness}$$

| Component | Computation | What it measures |
|-----------|-------------|-----------------|
| evidence | min(paired_stations / 15, 1) | Do we have enough paired stations? |
| classified | classified_rows / total_rows | Are spots properly classified F/B? |
| overlap | overlap_stations / total_stations | How many stations see both sectors? |
| cluster_quality | min(best_score / 5, 1) | How good is the best HDBSCAN cluster? |
| ci_tightness | max(0, 1 − CI_width / 10) | How narrow is the confidence interval? |

**Space weather penalty** (applied in Part 3): the base confidence is multiplied by
an ionospheric stability factor based on Kp and IMF Bz during the experiment.
Principle: *"cluster first, condition second."*

### Quality gate checks

| Gate | Fail | Warn | What it catches |
|------|------|------|-----------------|
| Front/back spots | < 30 | — | Not enough data in a sector |
| Unique RX per sector | < 10 | — | Too few receiving stations |
| Overlap (paired) stations | < 6 | < 10 | Weak F/B comparison basis |
| Paired stations (HDBSCAN) | < 5 | < 8 | Insufficient for clustering |
| Cluster CI width | — | > 8 dB | High uncertainty |
| Cluster sign consistency | — | < 60% | Ambiguous F/B direction |
| Kp stability | ≥ 6 | ≥ 5 or σ > 1.5 | Geomagnetic storm |
| IMF Bz stability | ≤ −10 nT | mean < −5 nT | Storm driving conditions |

In [ ]:
if len(pairs):
    n_pairs = len(pairs)
    total_spots = len(work[work['snr'].notna()])
    front_spots = len(work[(work['orientation']=='front') & work['snr'].notna()])
    back_spots = len(work[(work['orientation']=='back') & work['snr'].notna()])
    classified_spots = front_spots + back_spots
    front_rx = work.loc[work['orientation']=='front','rxCall'].nunique() if 'rxCall' in work.columns else 0
    back_rx = work.loc[work['orientation']=='back','rxCall'].nunique() if 'rxCall' in work.columns else 0
    total_rx = work['rxCall'].nunique() if 'rxCall' in work.columns else 0

    # Quality gate checks
    gate_checks = [
        ('Front spots ≥ 30',   front_spots >= 30,   front_spots >= 30,   f'{front_spots}/30'),
        ('Back spots ≥ 30',    back_spots >= 30,    back_spots >= 30,    f'{back_spots}/30'),
        ('Front RX ≥ 10',     front_rx >= 10,      front_rx >= 10,      f'{front_rx}/10'),
        ('Back RX ≥ 10',      back_rx >= 10,       back_rx >= 10,       f'{back_rx}/10'),
        ('Overlap stations ≥ 6 (fail) / ≥ 10 (warn)', n_pairs >= 6, n_pairs >= 10, f'{n_pairs}'),
        ('Paired stations ≥ 5 (fail) / ≥ 8 (warn)',   n_pairs >= 5, n_pairs >= 8,  f'{n_pairs}'),
        ('Total spots ≥ 80',  total_spots >= 80,   total_spots >= 80,   f'{total_spots}/80'),
    ]

    any_fail = not all(ok_fail for _, ok_fail, _, _ in gate_checks)
    any_warn = not all(ok_warn for _, _, ok_warn, _ in gate_checks)

    # HDBSCAN-specific checks
    hdbscan_checks = []
    if cluster_scores:
        best = cluster_scores[0]
        ci_width = best['bootstrap_ci_hi'] - best['bootstrap_ci_lo']
        sign_pct = best['sign_positive']
        hdbscan_checks.append(('Cluster CI width ≤ 8 dB', True, ci_width <= 8, f'{ci_width:.1f} dB'))
        hdbscan_checks.append(('Cluster sign+ ≥ 60%', True, sign_pct >= 0.6, f'{sign_pct*100:.0f}%'))

    # Confidence score
    if cluster_scores:
        evidence = min(1, n_pairs / 15)
        classified_ratio = classified_spots / max(total_spots, 1)
        overlap_ratio = n_pairs / max(total_rx, 1)
        best = cluster_scores[0]
        cluster_quality = min(1, best['score'] / 5.0)
        ci_width = best['bootstrap_ci_hi'] - best['bootstrap_ci_lo']
        ci_tightness = max(0, 1 - ci_width / 10)
        conf_score = 0.30 * evidence + 0.15 * classified_ratio + 0.15 * overlap_ratio + 0.25 * cluster_quality + 0.15 * ci_tightness
        formula_str = (f'0.30×{evidence:.2f} + 0.15×{classified_ratio:.2f} + 0.15×{overlap_ratio:.2f} '
                       f'+ 0.25×{cluster_quality:.2f} + 0.15×{ci_tightness:.2f} = {conf_score:.2f}')
        mode = 'HDBSCAN-aware'
    else:
        evidence = min(1, total_spots / 40)
        classified_ratio = classified_spots / max(total_spots, 1)
        overlap_ratio = n_pairs / max(total_rx, 1)
        conf_score = 0.50 * evidence + 0.30 * classified_ratio + 0.20 * overlap_ratio
        formula_str = f'0.50×{evidence:.2f} + 0.30×{classified_ratio:.2f} + 0.20×{overlap_ratio:.2f} = {conf_score:.2f}'
        mode = 'Legacy'

    if conf_score >= 0.72: conf_class = 'High'
    elif conf_score >= 0.45: conf_class = 'Moderate'
    else: conf_class = 'Low'

    gate_status = 'FAIL' if any_fail else ('WARN' if any_warn else 'PASS')
    confidence = 'Inconclusive' if any_fail else 'Provisional'

    print(f'=== Confidence Assessment ({mode}) ===')
    print(f'  Score: {conf_score*100:.0f}% ({conf_class})')
    print(f'  Formula: {formula_str}')
    print(f'  Quality gate: {gate_status}')
    print(f'  Inference label: {confidence}')
    print()
    print('Quality gate checks:')
    for label, ok_fail, ok_warn, detail in gate_checks:
        if not ok_fail:
            print(f'  ✗ FAIL  {label} — {detail}')
        elif not ok_warn:
            print(f'  ⚠ WARN  {label} — {detail}')
        else:
            print(f'  ✓ PASS  {label} — {detail}')
    for label, ok_fail, ok_warn, detail in hdbscan_checks:
        if not ok_fail:
            print(f'  ✗ FAIL  {label} — {detail}')
        elif not ok_warn:
            print(f'  ⚠ WARN  {label} — {detail}')
        else:
            print(f'  ✓ PASS  {label} — {detail}')

    if not any_fail:
        print()
        if cluster_scores:
            best = cluster_scores[0]
            print(f'  ★ Best estimate: {best["hodges_lehmann"]:+.1f} dB '
                  f'[{best["bootstrap_ci_lo"]:.1f}, {best["bootstrap_ci_hi"]:.1f}] '
                  f'(HDBSCAN cluster {best["cluster_id"]})')
        elif len(pairs):
            print(f'  Best estimate: {pairs["fb_delta"].median():+.1f} dB (station-paired median)')
        print(f'  → Replicate in another session to reach Strong confidence.')
    else:
        print(f'\n  → Collect more data to move beyond Inconclusive.')
    print()
    print('Note: Space weather ionospheric penalty is applied in Part 3 (maps & weather).')
else:
    print('No paired stations — cannot assess confidence.')

---

**Next:** Open **`03_maps_and_weather.ipynb`** for interactive maps of RX station locations,
space weather impact analysis, and the ionospheric stability confidence penalty.